# WIN_COUNTRY_CODE vs. ISO_COUNTRY_CODE — diagnostic (standalone)

Minimal, self-contained script to answer *Algorithms* reviewer #2's observation
that `WIN_COUNTRY_CODE` has far more unique values than `ISO_COUNTRY_CODE`
despite (presumably) covering the same set of countries. This only needs the
raw loaded data (`df_raw`) — no feature engineering, no model training — so
it should run in well under a minute.

Adjust `DRIVE_PARQUET` in the setup cell before running.


In [1]:
# ── COLAB SETUP ─────────────────────────────────────────────────────────
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

# ─── ADJUST this path to the folder where you uploaded the parquet in your Drive ───
DRIVE_PARQUET = '/content/drive/MyDrive/fci-datos/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
LOCAL_PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
# ────────────────────────────────────────────────────────────────────────────

if not os.path.exists(LOCAL_PARQUET):
    import shutil
    shutil.copy(DRIVE_PARQUET, LOCAL_PARQUET)
    print(f'Parquet copied to {LOCAL_PARQUET}')
else:
    print(f'Parquet already available at {LOCAL_PARQUET}')

print('Setup complete.')


Mounted at /content/drive
Parquet copied to /content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet
Setup complete.


In [2]:
import pandas as pd

df_raw = pd.read_parquet(LOCAL_PARQUET, columns=['ISO_COUNTRY_CODE', 'WIN_COUNTRY_CODE'])
print(f'Loaded: {df_raw.shape[0]:,} rows')


Loaded: 159,752 rows


In [3]:
# ── Diagnostic (Reviewer 2, Comment 7) ──────────────────────────────────
print('ISO_COUNTRY_CODE unique values:', df_raw['ISO_COUNTRY_CODE'].nunique())
print(sorted(df_raw['ISO_COUNTRY_CODE'].astype(str).unique()))
print()
print('WIN_COUNTRY_CODE unique values (raw):', df_raw['WIN_COUNTRY_CODE'].nunique())

# Normalize: strip whitespace, lowercase — checks if it's just formatting noise
win_norm = df_raw['WIN_COUNTRY_CODE'].astype(str).str.strip().str.lower()
iso_norm = df_raw['ISO_COUNTRY_CODE'].astype(str).str.strip().str.lower()

print('WIN_COUNTRY_CODE unique values (stripped+lowercased):', win_norm.nunique())

iso_set = set(iso_norm.unique())
win_not_in_iso = sorted(set(win_norm.unique()) - iso_set)
print(f'\nWIN_COUNTRY_CODE (normalized) values NOT present in ISO_COUNTRY_CODE set: {len(win_not_in_iso)}')
print(win_not_in_iso[:80])

# How many records are affected, and what do the extra values look like?
mask_extra = win_norm.isin(win_not_in_iso)
print(f'\nRecords with a WIN_COUNTRY_CODE value outside the ISO set: {mask_extra.sum():,} ({100*mask_extra.mean():.2f}%)')
print(df_raw.loc[mask_extra, 'WIN_COUNTRY_CODE'].value_counts().head(30))

# Cross-check: for records where the value IS in the ISO set after normalizing,
# does it actually match that row's own ISO_COUNTRY_CODE (same country), or is
# it a different (but valid) country -- i.e. a genuine cross-border award?
mask_valid = ~mask_extra
same_country = (win_norm[mask_valid] == iso_norm[mask_valid])
print(f'\nAmong records with a recognized WIN_COUNTRY_CODE, winner is in the SAME '
      f'country as the contracting authority: {same_country.mean()*100:.2f}%')
print(f'Winner is in a DIFFERENT (but valid) EU country: {(~same_country).mean()*100:.2f}%')


ISO_COUNTRY_CODE unique values: 30
['at', 'be', 'bg', 'cy', 'cz', 'de', 'dk', 'ee', 'es', 'fi', 'fr', 'gr', 'hr', 'hu', 'ie', 'it', 'lt', 'lu', 'lv', 'mk', 'mt', 'nl', 'no', 'pl', 'pt', 'ro', 'se', 'si', 'sk', 'uk']

WIN_COUNTRY_CODE unique values (raw): 677
WIN_COUNTRY_CODE unique values (stripped+lowercased): 677

WIN_COUNTRY_CODE (normalized) values NOT present in ISO_COUNTRY_CODE set: 648
['1a', 'ad', 'ae', 'ar---de---ca---sa---at---hu---hk---za', 'at---at', 'at---at---at', 'at---at---at---at', 'at---at---at---at---at', 'at---at---at---at---at---at', 'at---at---at---at---at---at---at', 'at---at---at---at---at---at---at---at---at---at---at', 'at---be', 'at---cz', 'at---de', 'at---fr', 'at---hu', 'at---hu---de', 'at---hu---hu', 'at---lt', 'au', 'be---be', 'be---be---be', 'be---be---be---be', 'be---be---be---be---be', 'be---be---be---be---be---be', 'be---be---be---be---be---be---be', 'be---be---be---be---be---be---be---be', 'be---be---be---pt', 'be---be---dk', 'be---be---uk', 'be---es